# TP2b — Réseau de neurones

Au TP2a, le modèle ne connaissait que la surface. Nous allons lui donner 4 caractéristiques :
la surface, le nombre de pièces, la latitude et la longitude, et remplacer la régression linéaire
par un **réseau de neurones**.

# Les données

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn

df = pd.read_csv('data/bordeaux_appartements_2025.csv.gz')

cols = {'valeur_fonciere': 'prix', 'surface_reelle_bati': 'surface', 'nombre_pieces_principales': 'nb_pieces'}
examples = df.rename(columns=cols, errors='raise')[['prix', 'surface', 'nb_pieces', 'latitude', 'longitude']]

features = ['surface', 'nb_pieces', 'latitude', 'longitude']
X = torch.tensor(examples[features].values, dtype=torch.float32)
Y = torch.tensor(examples[['prix']].values, dtype=torch.float32)
examples

,prix,surface,nb_pieces,latitude,longitude
0,455000.0,104.0,4.0,44.850132,-0.615116
1,393000.0,200.0,3.0,44.851554,-0.573621
2,186900.0,35.0,1.0,44.835897,-0.570661
3,620000.0,132.0,5.0,44.846194,-0.575516
4,100000.0,18.0,1.0,44.828206,-0.558298
...,...,...,...,...,...
3104,170000.0,63.0,3.0,44.864289,-0.578292
3105,189000.0,40.0,2.0,44.830659,-0.596693
3106,145665.0,26.0,1.0,44.833002,-0.570582
3107,110000.0,30.0,1.0,44.855053,-0.590581


## Plusieurs colonnes

`X` a maintenant **4 colonnes**, une par caractéristique.

In [2]:
print('X.shape :', X.shape)
print(X[:3])           # les 3 premières lignes : 3 appartements
print(X[:3, 0])        # la colonne 0 de ces 3 lignes : leurs surfaces

X.shape : torch.Size([3109, 4])
tensor([[104.0000,   4.0000,  44.8501,  -0.6151],
        [200.0000,   3.0000,  44.8516,  -0.5736],
        [ 35.0000,   1.0000,  44.8359,  -0.5707]])
tensor([104., 200.,  35.])


Pour standardiser, il faut **une moyenne par colonne** : on ne mélange pas des m² et des degrés de latitude.
C'est le rôle de `dim=0`.

In [3]:
print('X.mean()      :', X.mean())         # un seul nombre, toutes colonnes confondues : n'a pas de sens
print('X.mean(dim=0) :', X.mean(dim=0))    # 4 nombres : une moyenne par colonne

X.mean()      : tensor(26.5142)
X.mean(dim=0) : tensor([59.2332,  2.5600, 44.8439, -0.5803])


# Le modèle

Un réseau à **une couche cachée** de `M` neurones :

1. `self.hidden` : une couche cachée `nn.Linear`, avec 4 entrées et `M` sorties ;
2. `self.activation` : une fonction d'activation, `nn.ReLU()` par défaut ;
3. `self.output` : une couche de sortie `nn.Linear`, avec `M` entrées et 1 sortie (le prix).

Comme au TP2a, le modèle contient sa standardisation. Cette fois, il standardise aussi les **prix** :
le réseau calcule sur des petits nombres, et le modèle convertit sa sortie en euros.

La standardisation est déjà écrite. Complétez :

- dans `__init__`, les deux couches `nn.Linear` ;
- dans `forward`, les 3 étapes du réseau : `z` (couche cachée), `h` (activation) et `y` (couche de sortie).
  Chaque couche s'utilise comme une fonction : `self.hidden(x)`.

In [12]:
class NeuralNetworkModel(nn.Module):
    def __init__(self, X, Y, M, activation=nn.ReLU()):
        super().__init__()
        # une moyenne et un écart-type par colonne de X, puis pour les prix
        self.X_mean = X.mean(dim=0)
        self.X_std = X.std(dim=0)
        self.Y_mean = Y.mean()
        self.Y_std = Y.std()

        # TODO la couche cachée : 4 entrées, M sorties
        self.hidden =nn.Linear(4,M)
        self.activation = activation
        # TODO la couche de sortie : M entrées, 1 sortie
        self.output =nn.Linear(M,1)

    def forward(self, x):
        x = (x - self.X_mean) / self.X_std      # on standardise les caractéristiques
        # TODO les 3 étapes du réseau
        z = self.hidden(x)                             # couche cachée
        h = self.activation(z)                               # fonction d'activation
        y =self.output(h)                        # couche de sortie : un prix standardisé
        return y * self.Y_std + self.Y_mean     # on revient aux euros

In [13]:
torch.manual_seed(0)   # les paramètres sont tirés au hasard : la seed rend le résultat reproductible
model = NeuralNetworkModel(X, Y, M=30)
print(model)

NeuralNetworkModel(
  (hidden): Linear(in_features=4, out_features=30, bias=True)
  (activation): ReLU()
  (output): Linear(in_features=30, out_features=1, bias=True)
)


## Les shapes à travers le réseau

Faisons passer 5 appartements (standardisés) dans le réseau, couche par couche.

In [ ]:
x = (X[:5] - model.X_mean) / model.X_std
print(f'{"entrée":<55} {tuple(x.shape)}')
for layer in [model.hidden, model.activation, model.output]:
    x = layer(x)
    print(f'{str(layer):<55} {tuple(x.shape)}')

## Questions

1. Expliquez chaque shape affichée.
2. Combien de paramètres a ce réseau ? Détaillez le calcul (poids et biais de chaque couche),
   puis vérifiez avec la cellule ci-dessous.
3. La cellule `Q3` ci-dessous donne les 5 appartements à une couche `nn.Linear(3, 30)`. Exécutez-la et lisez
   le message d'erreur : que veut-il dire ? Comment faudrait-il écrire la couche pour que le calcul fonctionne ?

In [16]:
# Q2
print('nombre de paramètres :', sum(p.numel() for p in model.parameters()))

nombre de paramètres : 181


In [22]:
#Q3 : cette cellule provoque volontairement une erreur. Ne modifiez pas la classe NeuralNetworkModel.
 #Décommentez le code et exécutez
 wrong_layer = nn.Linear(3, 30)   # une couche qui attend 3 valeurs par appartement
 wrong_layer(X[:5])               # 5 appartements

IndentationError: unexpected indent (3538773671.py, line 3)

## Réponses

TODO
1.Chaque shape affiche entrée(5,4):5appartements avec charactéristiques
                     couches cachées(5,30)
                     RELU (5,30)
                     couche de sortie (5,1)
  2.Nombre de parametre
  Couche cachée : (4* 30) + 30 = 150
  Couche de sortie : (30 * 1) + 1 = 31
  Total : 150 + 31 = 181paramètres.
  3.erreur au niveau de nn.Linar(3,30) j'ai essayée en mettant (4,30) mais erreur toujours....


# L'entraînement

C'est la fonction `train()` du TP2a, avec deux changements :

- l'optimiseur **Adam** remplace SGD. C'est une descente de gradient qui adapte le pas de chaque paramètre,
  comme les deux learning rates du TP1b, mais automatiquement ;
- elle renvoie la RMSE de chaque epoch, pour tracer la courbe d'apprentissage.

In [24]:
def train(model, X, Y, epochs, alpha, verbose=True):
    cost_function = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=alpha)

    rmse = []
    for epoch in range(epochs):
        Y_pred = model(X)
        cost = cost_function(Y_pred, Y)
        cost.backward()
        optimizer.step()
        optimizer.zero_grad()

        rmse.append(cost.item() ** 0.5)
        if verbose and epoch % 100 == 0:
            print(f'epoch: {epoch}, RMSE: {rmse[-1]:,.0f} €')
    if verbose:
        print(f'RMSE finale : {rmse[-1]:,.0f} €')
    return rmse


def plot_learning_curve(rmse, skip=50):
    # skip : les premières epochs, qui écraseraient l'échelle verticale
    plt.plot(range(skip, len(rmse)), rmse[skip:])
    plt.xlabel('epoch')
    plt.ylabel('RMSE [€]')
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
rmse = train(model, X, Y, epochs=1000, alpha=0.01)
plot_learning_curve(rmse)

## Question

4. Au TP2a, la régression linéaire sur la surface donnait une RMSE de 76 986 €. Le réseau fait-il mieux ?

## Réponse

TODO


# L'inférence

Utilisons le modèle pour estimer le prix d'un appartement de **45 m², 2 pièces**, aux Chartrons.

Le modèle attend un tenseur de forme `(1, 4)` : une ligne (un appartement), et les 4 caractéristiques dans
l'ordre de `features` : surface, nombre de pièces, latitude, longitude.

In [ ]:
# Coordonnées (latitude, longitude) de quelques quartiers de Bordeaux
quartiers = {
    'Place de la Bourse': (44.8412, -0.5697),
    'Chartrons': (44.8545, -0.5720),
    'Bacalan': (44.8675, -0.5585),
    'La Bastide': (44.8425, -0.5530),
    'Caudéran': (44.8455, -0.6120),
    'Saint-Augustin': (44.8320, -0.5985),
    'Nansouty': (44.8215, -0.5765),
    'Gare Saint-Jean': (44.8258, -0.5562),
}

lat, lon = quartiers['Chartrons']
with torch.no_grad():
    # TODO le prix prédit pour 45 m², 2 pièces, à la position (lat, lon)
    price = ...

print(f'Prix estimé : {price.item():,.0f} €')

## Question

5. Estimez le prix du même appartement à Caudéran, puis à La Bastide. Quel est le quartier le plus cher ?

## Réponse

TODO


## Simulateur

Choisissez un quartier, une surface et un nombre de pièces : le modèle estime le prix.

In [ ]:
from ipywidgets import IntSlider, ToggleButtons, interact

# Boutons et curseur plutôt que listes déroulantes : dans PyCharm, la fenêtre d'une liste déroulante s'ouvre au mauvais endroit
style = {'description_width': 'initial'}


@interact(quartier=ToggleButtons(options=list(quartiers), value='Chartrons', description='Quartier', style=style),
          surface=IntSlider(min=15, max=150, step=5, value=45, description='Surface (m²)', style=style),
          nb_pieces=ToggleButtons(options=[1, 2, 3, 4, 5, 6], value=2, description='Pièces', style=style))
def estimate_price(quartier, surface, nb_pieces):
    lat, lon = quartiers[quartier]
    with torch.no_grad():
        price = model(torch.tensor([[float(surface), float(nb_pieces), lat, lon]]))
    print(f'Prix estimé : {price.item():,.0f} €')

# Les hyperparamètres

Le nombre de neurones `M`, le taux d'apprentissage `alpha`, la fonction d'activation et le nombre d'epochs
sont des **hyperparamètres** : c'est nous qui les choisissons, pas l'entraînement.

`run_experiment()` construit un `NeuralNetworkModel`, l'entraîne, et ajoute une ligne au tableau `results`.

In [ ]:
results = []

def run_experiment(M, alpha, activation=nn.ReLU(), epochs=1000):
    torch.manual_seed(0)
    model = NeuralNetworkModel(X, Y, M, activation)
    rmse = train(model, X, Y, epochs, alpha, verbose=False)
    results.append({'M': M, 'alpha': alpha, 'activation': type(activation).__name__,
                    'epochs': epochs, 'RMSE (€)': round(rmse[-1])})
    return pd.DataFrame(results)

## Questions

6. En partant de la configuration ci-dessous, changez **un seul hyperparamètre à la fois** :
   - `alpha` : 1, 0.1, 0.001, 0.0001
   - `M` : 5, 100, 300
   - `activation` : `nn.Tanh()`, `nn.Sigmoid()`

   Quel hyperparamètre change le plus la RMSE ?
7. Avec `M=100`, augmentez `epochs` : 2 000, puis 5 000.
   Le coût finit-il par cesser de baisser ? D'après vous, ce modèle sera-t-il aussi bon sur des appartements
   qu'il n'a jamais vus ?

In [ ]:
# TODO questions 6 et 7 : ajoutez un appel à run_experiment() par essai
# chaque appel ajoute une ligne dans le tableau
run_experiment(M=30, alpha=0.01)


## Réponses

TODO


# Ce que le réseau a appris

On fixe un appartement de 50 m², 3 pièces, et on demande au modèle son prix partout dans Bordeaux.
La carte montre ce que le réseau a appris de la **localisation**.

In [ ]:
from carte_prix import trace_carte_prix

trace_carte_prix(model, examples, surface=50, nb_pieces=3)
plt.show()

## Question

8. Le réseau n'a reçu que des latitudes et des longitudes, jamais de plan de la ville.
   A-t-il trouvé où sont les quartiers chers ?

## Réponse

TODO


# Bonus - Deux couches cachées

9. Copiez la classe `NeuralNetworkModel` dans une nouvelle classe `DeepNeuralNetworkModel`, et ajoutez une deuxième
   couche cachée `self.hidden2` de `M` entrées et `M` sorties, suivie de l'activation.
   Combien de paramètres pour `M=30` ? La RMSE est-elle meilleure qu'avec une seule couche cachée de même largeur ?

In [ ]:
# TODO question 9


## Réponse

TODO
